In [79]:
import pandas as pd
import numpy as np 
from tmdbv3api import Movie, Person, TMDb
import requests


path_to_data = '../data/2024_Big5_dataset.csv'



In [114]:
def Oscar_df(path):
    '''
    Function reads in a csv containing data on oscar nominees and converts into a pandas dataframe
    
    Param: path of csv file
    Returns: pandas dataframe
    
    '''

    #Take in oscar nominee csv
    oscar_df = pd.read_csv(path)

    #select for all columns that contain strings
    str_cols = oscar_df.select_dtypes('str').columns
    #convert all rows of each column with str type to lowercase
    oscar_df[str_cols] = oscar_df[str_cols].apply(lambda x: x.str.lower())
    

    return oscar_df

In [ ]:
class category: 

    def __init__(self, df, category):
        
        self.category = category
        self.df = df[df['category'].str.contains(self.category)]
        self.movie_list = list(self.df['film'])


In [131]:
class nominee: 

    def __init__(self, name, df, cat, job):

        self.name = name
        self.df = df[df['film'].isin(self._get_filmography())]
        c = category(df, cat)
        #self.cat = cat
        self.job = job
        # Isolate only the oscar movies the nominee has appeared in for this category (even if not the awards intended recipient)
        self.nom_category_appearance = self.df[
            self.df['film'].isin(c.df['film']) & 
            self.df['category'].isin(c.df['category'])
        ]
        

    def _get_filmography(self):

        tmdb = TMDb()
        person = Person()
        movie = Movie()

        tmdb.api_key = 'f49d2ebf0d11031312ade120f61c513d'

        nominee_search = person.search(self.name)

        nomID = nominee_search[0].id

        url = f"https://api.themoviedb.org/3/person/{nomID}/combined_credits"
        params = {"api_key": tmdb.api_key}
        
        data = requests.get(url, params=params).json()

        movies = data['cast']

        filmography = [m["title"] for m in movies if "title" in m]

        filmography = [f.lower() for f in filmography]

        return filmography
    
        

        

    def oscars_score(self):

        '''
        function calculates a weight based on oscar nominations and wins. Wins are heavily favored, but nominations are not counted against as they would be in a probabilistic determination 
        of success 
        
        param: self <- object of nominee class
        returns: weighted oscar score
        
        '''
        # Isolate only the winners from the category nominations the nominee has appeared in 
        outcomes = list(self.nom_category_appearance['winner'])
        print(outcomes)
        # initialize oscar score
        oscar_score = 0

        # If nominee has no previous nominations for category
        if len(outcomes) == 0:
            return 0
        else:
            # Iterate through each movie's outcome and add weighted score to oscar score
            for decision in outcomes:
                if decision == True: 
                    #arbitrary weighting, favorable weighting for a win
                    oscar_score += 3
                else: 
                    #arbitrary weighting, less favorable but still positive weighting for a nomination
                    oscar_score += 1

        return oscar_score

    def synergy_boost(self, crew_list):

        '''
        Function considers past professional relationships with other nominees and calculates a synergy boost.

        param1: self <- object of nominee class
        param2: crew_list <- list of other crew members on movie, each item in list is an instance of the nominee class

        returns: new score with added synergy boost
        '''
        
        # Initialize empy list for common movies
        all_common_movies = []

        # Initialize synergy score
        synergy_score = 0 

        # Iterate through crew list
        for member in crew_list:
            # Exclude self
            if member.name != self.name:
                # Takes the intersection between self and other crew members filmoraphy
                common_movies = set(member._get_filmography) & set(self._get_filmography)
                # if 1 or more movies in common
                if len(common_movies) > 0:
                    # iterate through common movies
                    for movies in common_movies: 
                        # add to list of common movies shared between self and all other nominees 
                        all_common_movies.append(movies)        
                else:
                    # If no movies in common return a synergy score of 0
                    return 0
                
        # iterate through list of all shared movies
        for movie in all_common_movies: 
            # If the movie won an oscar, add weight
            if movie in self.df[self.df['film'] == movie] & (self.df['winner'] == True):
                synergy_score += 0.6
            # If the movie was nominated but didn't win, add weight
            elif movie in self.df[self.df['film'] == movie] & (self.df['winner'] == False):
                synergy_score += 0.2
            # If movie wasn't nominated for some reason, pass to next movie
            else: 
                pass

        return synergy_score

In [83]:
role_dict = {
    'directing': 'director', 
    'actor': 'actor',
    'actress': 'actress',
    'writing': 'writer'
}

oscar_df = Oscar_df(path_to_data)

role_nom_dict = oscar_df.groupby('category')['name'].apply(list).to_dict()

In [136]:
class rand_movie: 

    def __init__(self, category, role_title, poss_nom_dict):

        self.category = category
        self.role_title = role_title
        self.role_nom = poss_nom_dict

        for keys in poss_nom_dict.keys():
            if keys in self.category:
                self.avail_roles = poss_nom_dict[keys]
        


    #def get_crew (self, df):

movie1 = rand_movie('directing', 'director', role_nom_dict)
#movie1.avail_roles
movie1.role_nom

        

         
        
        


         

         

        

        

{'actor in a leading role': ['adrien brody',
  'timothée chalamet',
  'colman domingo',
  'ralph fiennes',
  'sebastian stan'],
 'actor in a supporting role': ['yura borisov',
  'kieran culkin',
  'edward norton',
  'guy pearce',
  'jeremy strong'],
 'actress in a leading role': ['cynthia erivo',
  'karla sofía gascón',
  'mikey madison',
  'demi moore',
  'fernanda torres'],
 'actress in a supporting role': ['monica barbaro',
  'ariana grande',
  'felicity jones',
  'isabella rossellini',
  'zoe saldaña'],
 'best picture': ['alex coco/samantha quan/sean baker',
  'nick gordon/brian young/andrew morrison/d.j. gugenheim/brady corbet',
  'fred berger/james mangold/alex heineman',
  'tessa ross/juliette howell/michael a. jackman',
  'mary parent/cale boyter/tanya lapointe/denis villeneuve',
  'pascal caucheteux/jacques audiard',
  'maria carlota bruno/rodrigo teixeira',
  'dede gardner/jeremy kleiner/joslyn barnes',
  'coralie fargeat/tim bevan/eric fellner',
  'marc platt'],
 'directing'

In [ ]:
demi = nominee('demi moore', oscar_df, 'directing', 'actress')
direct = category(oscar_df, 'directing')
#david_lynch._get_filmography()

#demi.oscars_score()
direct.df


mikey = nominee('mikey madison', oscar_df, 'directing', 'actress')

#mikey.df
#mikey.nom_category_appearance
#mikey.oscars_score()

[True]


3